In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Colab_notebook/NLP-Powered Business Review Intelligence System/Reviews.csv')
print("Total reviews:", df.shape[0])

Total reviews: 568454


In [ ]:
print("Columns:", df.columns.tolist())

Columns: ['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']


In [ ]:
print("\n", df.head(3))


    Id   ProductId          UserId                      ProfileName  \
0   1  B001E4KFG0  A3SGXH7AUHU8GW                       delmartian   
1   2  B00813GRG4  A1D87F6ZCVE5NK                           dll pa   
2   3  B000LQOCH0   ABXLMWJIXXAIN  Natalia Corres "Natalia Corres"   

   HelpfulnessNumerator  HelpfulnessDenominator  Score        Time  \
0                     1                       1      5  1303862400   
1                     0                       0      1  1346976000   
2                     1                       1      4  1219017600   

                 Summary                                               Text  
0  Good Quality Dog Food  I have bought several of the Vitality canned d...  
1      Not as Advertised  Product arrived labeled as Jumbo Salted Peanut...  
2  "Delight" says it all  This is a confection that has been around a fe...  


In [ ]:
print("\nMissing values:\n", df.isnull().sum())


Missing values:
 Id                         0
ProductId                  0
UserId                     0
ProfileName               26
HelpfulnessNumerator       0
HelpfulnessDenominator     0
Score                      0
Time                       0
Summary                   27
Text                       0
dtype: int64


In [ ]:
print("\nStar rating counts:\n", df['Score'].value_counts())


Star rating counts:
 Score
5    363122
4     80655
1     52268
3     42640
2     29769
Name: count, dtype: int64


In [ ]:
import pandas as pd

# ---- Total number of rows in dataset ----
total_rows = len(df)
print("Total rows:", total_rows)

# ---- Count of null values in each column ----
null_count = df.isnull().sum()

# ---- Percentage of null values in each column ----
# Formula: (null count / total rows) * 100
null_percentage = (null_count / total_rows) * 100

# ---- Combine both into one clean table ----
null_report = pd.DataFrame({
    'Null Count'      : null_count,
    'Null Percentage' : null_percentage.round(2)  # round to 2 decimal places
})

# ---- Show only columns that actually HAVE null values ----
null_report = null_report[null_report['Null Count'] > 0]

# ---- Final output ----
if null_report.empty:
    print("\n No null values found in the dataset!")
else:
    print("\n Null Value Report:")
    print(null_report)

Total rows: 568454

 Null Value Report:
             Null Count  Null Percentage
ProfileName          26              0.0
Summary              27              0.0


In [ ]:
# Keep only the 3 useful columns
df = df[['Text', 'Summary', 'Score']]

# Remove any empty reviews
df = df.dropna(subset=['Text'])

# Remove duplicate reviews
df = df.drop_duplicates(subset=['Text'])

print("Clean dataset size:", df.shape[0])
print(df.head())

Clean dataset size: 393579
                                                Text                Summary  \
0  I have bought several of the Vitality canned d...  Good Quality Dog Food   
1  Product arrived labeled as Jumbo Salted Peanut...      Not as Advertised   
2  This is a confection that has been around a fe...  "Delight" says it all   
3  If you are looking for the secret ingredient i...         Cough Medicine   
4  Great taffy at a great price.  There was a wid...            Great taffy   

   Score  
0      5  
1      1  
2      4  
3      2  
4      5  


#Add a Sentiment label column

In [ ]:
# Convert star rating into sentiment
# 1,2 stars = Negative | 3 stars = Neutral | 4,5 stars = Positive

def label_sentiment(score):
    if score <= 2:
        return 'Negative'
    elif score == 3:
        return 'Neutral'
    else:
        return 'Positive'

df['Sentiment'] = df['Score'].apply(label_sentiment)

print(df['Sentiment'].value_counts())

Sentiment
Positive    306758
Negative     57067
Neutral      29754
Name: count, dtype: int64


# Cleaning the review text for NLP


In [ ]:
import nltk
import re
import string


nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('punkt')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# English stopwords list (the, is, a, an, etc.)
stop_words = set(stopwords.words('english'))

print("Libraries ready!")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


Libraries ready!


#  Build your text cleaning function

In [ ]:
def clean_text(text):

    # Step 1: Convert to lowercase
    # "GREAT Product" → "great product"
    text = text.lower()

    # Step 2: Remove HTML tags like <br>, <b>
    # some reviews have html inside them
    text = re.sub(r'<.*?>', '', text)

    # Step 3: Remove URLs
    text = re.sub(r'http\S+', '', text)

    # Step 4: Remove punctuation like !@#$%^&*
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Step 5: Remove numbers
    text = re.sub(r'\d+', '', text)

    # Step 6: Tokenize — split sentence into individual words
    # "good pizza taste" → ["good", "pizza", "taste"]
    tokens = word_tokenize(text)

    # Step 7: Remove stopwords — useless words like the, is, a
    tokens = [word for word in tokens if word not in stop_words]

    # Step 8: Remove very short words (1-2 characters)
    tokens = [word for word in tokens if len(word) > 2]

    # Step 9: Join tokens back into a clean sentence
    cleaned = ' '.join(tokens)

    return cleaned

# Apply cleaning to entire dataset

In [ ]:
import nltk
print("Cleaning reviews... this may take 1-2 minutes")
nltk.download('punkt_tab', quiet=True)
df['Cleaned_Text'] = df['Text'].apply(clean_text)

print("Done!")
print("\nOriginal review:\n", df['Text'][0])
print("\nCleaned review:\n", df['Cleaned_Text'][0])

Cleaning reviews... this may take 1-2 minutes
Done!

Original review:
 I have bought several of the Vitality canned dog food products and have found them all to be of good quality. The product looks more like a stew than a processed meat and it smells better. My Labrador is finicky and she appreciates this product better than  most.

Cleaned review:
 bought several vitality canned dog food products found good quality product looks like stew processed meat smells better labrador finicky appreciates product better


#Extract Bigrams and Trigrams

In [ ]:
from nltk import bigrams, trigrams
from collections import Counter

# Ensure 'Cleaned_Text' column is present before proceeding
# This makes the cell robust if the previous cleaning step wasn't fully reflected
# Assumes 'clean_text' function is defined and 'df' and 'df['Text']' are available
if 'Cleaned_Text' not in df.columns:
    print("Warning: 'Cleaned_Text' column not found. Attempting to create it now.")
    df['Cleaned_Text'] = df['Text'].apply(clean_text)

# Combine all cleaned reviews into one big list of words
all_words = ' '.join(df['Cleaned_Text']).split()

# Bigrams = 2 word combinations ("battery life", "fast delivery")
bigram_list  = list(bigrams(all_words))
top_bigrams  = Counter(bigram_list).most_common(10)

# Trigrams = 3 word combinations ("great battery life", "next day delivery")
trigram_list = list(trigrams(all_words))
top_trigrams = Counter(trigram_list).most_common(10)

print("Top 10 Bigrams:")
for bg, count in top_bigrams:
    print(f"  {bg[0]} {bg[1]}  →  {count} times")

print("\nTop 10 Trigrams:")
for tg, count in top_trigrams:
    print(f"  {tg[0]} {tg[1]} {tg[2]}  →  {count} times")

Top 10 Bigrams:
  taste like  →  10024 times
  highly recommend  →  9697 times
  grocery store  →  9222 times
  peanut butter  →  9017 times
  gluten free  →  8400 times
  green tea  →  8249 times
  ive tried  →  8108 times
  tastes like  →  8008 times
  much better  →  7993 times
  dont know  →  6777 times

Top 10 Trigrams:
  local grocery store  →  2251 times
  high fructose corn  →  1424 times
  fructose corn syrup  →  1351 times
  highly recommend product  →  1343 times
  would highly recommend  →  1332 times
  bobs red mill  →  1208 times
  health food store  →  1198 times
  goes long way  →  1026 times
  would recommend anyone  →  858 times
  doesnt taste like  →  841 times


# Saving cleaned dataset

In [ ]:
# Save to Drive so you never have to clean again
df.to_csv('/content/drive/MyDrive/Reviews_Cleaned.csv', index=False)

print("Saved! Your cleaned file is in Google Drive")
print("Final dataset shape:", df.shape)
print(df[['Text', 'Cleaned_Text', 'Sentiment']].head(3))

Saved! Your cleaned file is in Google Drive
Final dataset shape: (393579, 5)
                                                Text  \
0  I have bought several of the Vitality canned d...   
1  Product arrived labeled as Jumbo Salted Peanut...   
2  This is a confection that has been around a fe...   

                                        Cleaned_Text Sentiment  
0  bought several vitality canned dog food produc...  Positive  
1  product arrived labeled jumbo salted peanutsth...  Negative  
2  confection around centuries light pillowy citr...  Positive  


In [ ]:
print("\n", df.head(3))


                                                 Text                Summary  \
0  I have bought several of the Vitality canned d...  Good Quality Dog Food   
1  Product arrived labeled as Jumbo Salted Peanut...      Not as Advertised   
2  This is a confection that has been around a fe...  "Delight" says it all   

   Score Sentiment                                       Cleaned_Text  
0      5  Positive  bought several vitality canned dog food produc...  
1      1  Negative  product arrived labeled jumbo salted peanutsth...  
2      4  Positive  confection around centuries light pillowy citr...  


#  Building the CNN Sentiment Model

In [ ]:
# Install and import everything
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

print("TensorFlow version:", tf.__version__)
print("All libraries loaded!")

TensorFlow version: 2.19.0
All libraries loaded!


# Preparing data for CNN

In [ ]:
# Use only 100k reviews to keep training fast on free Colab
df_model = df.sample(100000, random_state=42)

# Input = cleaned text
X = df_model['Cleaned_Text'].astype(str)

# Output = sentiment label
y = df_model['Sentiment']

# Convert Positive/Negative/Neutral into numbers
# Positive=2, Neutral=1, Negative=0
le = LabelEncoder()
y = le.fit_transform(y)

print("Classes:", le.classes_)
print("Sample labels:", y[:5])

Classes: ['Negative' 'Neutral' 'Positive']
Sample labels: [2 1 2 2 2]


# Tokenize and pad the text

#### Re-initializing Tokenizer to ensure it's in memory for saving.

In [ ]:
# Maximum number of unique words to keep
MAX_WORDS = 20000

# Maximum length of each review (words)
MAX_LEN = 100

# Tokenizer converts words to numbers
# "good pizza" → [45, 231]
tokenizer = Tokenizer(num_words=MAX_WORDS)
tokenizer.fit_on_texts(X)
X_seq = tokenizer.texts_to_sequences(X)

# Pad so every review is same length
# Short reviews get zeros added, long ones get trimmed
X_pad = pad_sequences(X_seq, maxlen=MAX_LEN)

print("Shape after padding:", X_pad.shape)
print("Sample tokenized review:", X_pad[0])

Shape after padding: (100000, 100)
Sample tokenized review: [   0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0   28   79  241   72  287  242  829   26 1752
  471    5    6  445 3555   53  287 1671  738 4094  277  646   83  153
   36    1  927  135   46   31  129  287 8774  477  414 1609  241   72
  264    6]


In [ ]:
# Maximum number of unique words to keep
MAX_WORDS = 20000

# Maximum length of each review (words)
MAX_LEN = 100

# Tokenizer converts words to numbers
# "good pizza" → [45, 231]
tokenizer = Tokenizer(num_words=MAX_WORDS)
tokenizer.fit_on_texts(X)
X_seq = tokenizer.texts_to_sequences(X)

# Pad so every review is same length
# Short reviews get zeros added, long ones get trimmed
X_pad = pad_sequences(X_seq, maxlen=MAX_LEN)

print("Shape after padding:", X_pad.shape)
print("Sample tokenized review:", X_pad[0])

Shape after padding: (100000, 100)
Sample tokenized review: [   0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0   28   79  241   72  287  242  829   26 1752
  471    5    6  445 3555   53  287 1671  738 4094  277  646   83  153
   36    1  927  135   46   31  129  287 8774  477  414 1609  241   72
  264    6]


#  Split into train and test

In [ ]:
# 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X_pad, y, test_size=0.2, random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 80000
Testing samples: 20000


# Building the CNN model

In [ ]:
model = Sequential([

    # Layer 1: Embedding — converts word numbers into vectors
    # Think of it as giving each word a meaning in numbers
    Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LEN),

    # Layer 2: CNN — finds patterns in word sequences
    # Like finding "not good" or "very bad" as a pattern
    Conv1D(filters=128, kernel_size=5, activation='relu'),

    # Layer 3: Pooling — picks the most important patterns
    GlobalMaxPooling1D(),

    # Layer 4: Dense — learns from those patterns
    Dense(64, activation='relu'),

    # Layer 5: Dropout — prevents overfitting
    Dropout(0.3),

    # Layer 6: Output — 3 classes (Positive, Neutral, Negative)
    Dense(3, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# Train the model

In [ ]:
print("Training started... takes about 3-5 minutes on GPU")

history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=256,
    validation_split=0.1,
    verbose=1
)

print("\nTraining Done!")

Training started... takes about 3-5 minutes on GPU
Epoch 1/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 17s 25ms/step - accuracy: 0.8180 - loss: 0.5188 - val_accuracy: 0.8371 - val_loss: 0.4364
Epoch 2/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8678 - loss: 0.3616 - val_accuracy: 0.8526 - val_loss: 0.3983
Epoch 3/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9083 - loss: 0.2495 - val_accuracy: 0.8466 - val_loss: 0.4476
Epoch 4/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.9549 - loss: 0.1363 - val_accuracy: 0.8459 - val_loss: 0.5428
Epoch 5/5
282/282 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9819 - loss: 0.0619 - val_accuracy: 0.8381 - val_loss: 0.6598

Training Done!


# Test the accuracy

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {accuracy*100:.2f}%")

# Test with a real example
sample = ["This product is absolutely amazing, best I ever bought"]
sample_seq = tokenizer.texts_to_sequences(sample)
sample_pad = pad_sequences(sample_seq, maxlen=MAX_LEN)
prediction = model.predict(sample_pad)
predicted_class = le.classes_[np.argmax(prediction)]
print(f"\nSample review: {sample[0]}")
print(f"Predicted sentiment: {predicted_class}")

Test Accuracy: 82.92%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 444ms/step

Sample review: This product is absolutely amazing, best I ever bought
Predicted sentiment: Positive


   # MetricYour     ScoreMeaningTest      Accuracy
    83.12%       Model predicts correctly 83 out of 100 reviews Epoch 5 Training
    97.58%       Model learned the training data very well
    Sample prediction       PositiveWorking correctly!

#Topic Modelling with LDA
##This finds hidden themes across all reviews automatically.

In [ ]:
!pip install gensim
import gensim
from gensim import corpora
from gensim.models import LdaModel
from nltk.tokenize import word_tokenize

print("Gensim version:", gensim.__version__)
print("Libraries ready!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 24.7 MB/s eta 0:00:00
Gensim version: 4.4.0
Libraries ready!


#  Prepare text for LDA

In [ ]:
# Tokenize cleaned text into list of words
# LDA needs words as a list, not a sentence
# "good pizza taste" → ["good", "pizza", "taste"]

print("Preparing tokens...")

tokenized_reviews = df['Cleaned_Text'].astype(str).apply(
    lambda x: x.split()
).tolist()

# Build dictionary — maps every unique word to a number
dictionary = corpora.Dictionary(tokenized_reviews)

# Remove words that appear in less than 5 reviews (too rare)
# Remove words that appear in more than 50% of reviews (too common)
dictionary.filter_extremes(no_below=5, no_above=0.5)

# Convert each review into bag of words format
# [(word_id, word_count), (word_id, word_count)...]
corpus = [dictionary.doc2bow(review) for review in tokenized_reviews]

print("Dictionary size:", len(dictionary))
print("Corpus ready! Total reviews:", len(corpus))

Preparing tokens...
Dictionary size: 43757
Corpus ready! Total reviews: 393579


#  Train LDA model

In [ ]:
import random

# Use only 50k reviews instead of 393k
# LDA doesn't need full dataset to find topics
df_lda = df.sample(50000, random_state=42)

# Tokenize
tokenized_lda = df_lda['Cleaned_Text'].astype(str).apply(
    lambda x: x.split()
).tolist()

# Build dictionary
from gensim import corpora
from gensim.models import LdaMulticore

dictionary = corpora.Dictionary(tokenized_lda)
dictionary.filter_extremes(no_below=5, no_above=0.5)
corpus = [dictionary.doc2bow(review) for review in tokenized_lda]

print("Dictionary size:", len(dictionary))
print("Training LDA on 50k reviews...")

# LdaMulticore is 3x faster than LdaModel
lda_model = LdaMulticore(
    corpus=corpus,
    id2word=dictionary,
    num_topics=5,
    random_state=42,
    passes=5,        # reduced from 10 to 5
    workers=2        # uses multiple CPU cores
)

print("LDA Training Done!")

Dictionary size: 13730
Training LDA on 50k reviews...
LDA Training Done!


In [ ]:
print(df.head())

                                                Text                Summary  \
0  I have bought several of the Vitality canned d...  Good Quality Dog Food   
1  Product arrived labeled as Jumbo Salted Peanut...      Not as Advertised   
2  This is a confection that has been around a fe...  "Delight" says it all   
3  If you are looking for the secret ingredient i...         Cough Medicine   
4  Great taffy at a great price.  There was a wid...            Great taffy   

   Score Sentiment                                       Cleaned_Text  
0      5  Positive  bought several vitality canned dog food produc...  
1      1  Negative  product arrived labeled jumbo salted peanutsth...  
2      4  Positive  confection around centuries light pillowy citr...  
3      2  Negative  looking secret ingredient robitussin believe f...  
4      5  Positive  great taffy great price wide assortment yummy ...  


#  See what topics were discovered

In [ ]:
print("Top 5 Topics discovered in your reviews:\n")

for idx, topic in lda_model.print_topics(num_words=8):
    print(f"Topic {idx + 1}:")

    # Clean up the output to show just words
    words = [word.split('*')[1].replace('"','').strip()
             for word in topic.split('+')]
    print(f"  Keywords: {', '.join(words)}")
    print()

Top 5 Topics discovered in your reviews:

Topic 1:
  Keywords: food, dog, one, like, dogs, treats, would, get

Topic 2:
  Keywords: like, taste, chocolate, sugar, flavor, good, sweet, product

Topic 3:
  Keywords: like, good, taste, great, flavor, one, make, use

Topic 4:
  Keywords: amazon, product, great, price, find, good, buy, order

Topic 5:
  Keywords: coffee, tea, like, flavor, good, taste, cup, one



# Assign a topic to every review

In [ ]:
# Redefine the function that was lost on disconnect
def get_dominant_topic(review_bow):
    # Get topic distribution for one review
    topics = lda_model.get_document_topics(review_bow)
    # Return the topic with highest probability
    if topics:
        dominant = max(topics, key=lambda x: x[1])
        return dominant[0]
    return -1

print("Function defined!")

Function defined!


In [ ]:
# Since we trained LDA on only 50k reviews
# we assign topics only to those 50k rows

print("Assigning topics to reviews...")

# Get the first 50k rows of df to match corpus size
df_lda_subset = df.iloc[:50000].copy()

# Assign dominant topic to each of the 50k reviews
df_lda_subset['Topic'] = [
    get_dominant_topic(bow) for bow in corpus
]

print("Done!")
print("\nTopic distribution:")
print(df_lda_subset['Topic'].value_counts())

# Merge topic back into main df
df = df.merge(
    df_lda_subset[['Topic']],
    left_index=True,
    right_index=True,
    how='left'
)

# Fill remaining rows with -1 (no topic assigned)
df['Topic'] = df['Topic'].fillna(-1).astype(int)

print("\nFull dataset shape:", df.shape)
print(df[['Cleaned_Text', 'Sentiment', 'Topic']].head(5))

Assigning topics to reviews...
Done!

Topic distribution:
Topic
3    12859
2    11698
1     9569
4     8793
0     7081
Name: count, dtype: int64

Full dataset shape: (393579, 6)
                                        Cleaned_Text Sentiment  Topic
0  bought several vitality canned dog food produc...  Positive      2
1  product arrived labeled jumbo salted peanutsth...  Negative      0
2  confection around centuries light pillowy citr...  Positive      3
3  looking secret ingredient robitussin believe f...  Negative      4
4  great taffy great price wide assortment yummy ...  Positive      3


# Save updated dataset

In [ ]:
df.to_csv('/content/drive/MyDrive/Reviews_Cleaned.csv', index=False)
print("Saved with topics!")
print(df[['Cleaned_Text', 'Sentiment', 'Topic']].head(5))

Saved with topics!
                                        Cleaned_Text Sentiment  Topic
0  bought several vitality canned dog food produc...  Positive      2
1  product arrived labeled jumbo salted peanutsth...  Negative      0
2  confection around centuries light pillowy citr...  Positive      3
3  looking secret ingredient robitussin believe f...  Negative      4
4  great taffy great price wide assortment yummy ...  Positive      3


# RNN/LSTM Rating Predictor
##This predicts the star rating (1-5) just from the review text.

In [ ]:
from tensorflow.keras.layers import LSTM, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np

print("Libraries ready!")

Libraries ready!


#   Prepare data for RNN

In [ ]:
# Input = cleaned text
X_rnn = df['Cleaned_Text'].astype(str)

# Output = star rating (1 to 5)
y_rnn = df['Score'].values - 1
# Minus 1 because model needs 0-4 not 1-5
# 1 star → 0, 2 stars → 1 ... 5 stars → 4

# Tokenize text into sequences
X_rnn_seq = tokenizer.texts_to_sequences(X_rnn)

# Pad all reviews to same length
X_rnn_pad = pad_sequences(X_rnn_seq, maxlen=MAX_LEN)

# Split into train and test
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_rnn_pad, y_rnn,
    test_size=0.2,
    random_state=42
)

print("Training samples:", X_train_r.shape[0])
print("Testing samples:", X_test_r.shape[0])
print("Sample labels (0=1star, 4=5star):", y_train_r[:5])

Training samples: 314863
Testing samples: 78716
Sample labels (0=1star, 4=5star): [4 4 4 4 4]


# Build RNN/LSTM model

In [ ]:
rnn_model = Sequential([

    # Layer 1: Embedding — word to vector
    Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LEN),

    # Layer 2: Bidirectional LSTM
    # Reads review forwards AND backwards
    # Like reading "not good at all" normally
    # AND backwards to understand context better
    Bidirectional(LSTM(64, return_sequences=True)),

    # Layer 3: Second LSTM layer
    # Learns deeper patterns from first LSTM
    Bidirectional(LSTM(32)),

    # Layer 4: Dense layer
    Dense(64, activation='relu'),

    # Layer 5: Dropout — prevents overfitting
    Dropout(0.3),

    # Layer 6: Output — 5 classes (1 to 5 stars)
    Dense(5, activation='softmax')
])

rnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

rnn_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

#Train RNN model

In [ ]:
# Early stopping — stops training if model stops improving
# Prevents wasting time on unnecessary epochs
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,           # stop after 2 epochs of no improvement
    restore_best_weights=True
)

print("Training RNN... takes about 5-8 minutes on GPU")

rnn_history = rnn_model.fit(
    X_train_r, y_train_r,
    epochs=5,
    batch_size=256,
    validation_split=0.1,
    callbacks=[early_stop],
    verbose=1
)

print("\nRNN Training Done!")

Training RNN... takes about 5-8 minutes on GPU
Epoch 1/5
1107/1107 ━━━━━━━━━━━━━━━━━━━━ 47s 32ms/step - accuracy: 0.6995 - loss: 0.8255 - val_accuracy: 0.7162 - val_loss: 0.7569
Epoch 2/5
1107/1107 ━━━━━━━━━━━━━━━━━━━━ 35s 32ms/step - accuracy: 0.7325 - loss: 0.7166 - val_accuracy: 0.7244 - val_loss: 0.7479
Epoch 3/5
1107/1107 ━━━━━━━━━━━━━━━━━━━━ 36s 32ms/step - accuracy: 0.7501 - loss: 0.6636 - val_accuracy: 0.7206 - val_loss: 0.7531
Epoch 4/5
1107/1107 ━━━━━━━━━━━━━━━━━━━━ 35s 32ms/step - accuracy: 0.7685 - loss: 0.6131 - val_accuracy: 0.7144 - val_loss: 0.7839

RNN Training Done!


#Test accuracy and predict

In [ ]:
# Overall accuracy
loss, accuracy = rnn_model.evaluate(X_test_r, y_test_r, verbose=0)
print(f"RNN Test Accuracy: {accuracy*100:.2f}%")

# Test with real examples
test_reviews = [
    "This product is absolutely terrible, complete waste of money",
    "Amazing quality, exceeded all my expectations, will buy again",
    "It is okay, nothing special but does the job"
]

for review in test_reviews:
    seq = tokenizer.texts_to_sequences([review])
    pad = pad_sequences(seq, maxlen=MAX_LEN)
    pred = rnn_model.predict(pad, verbose=0)
    stars = np.argmax(pred) + 1  # add 1 back to get 1-5
    print(f"\nReview: {review[:50]}...")
    print(f"Predicted Rating: {stars} stars")

RNN Test Accuracy: 72.55%

Review: This product is absolutely terrible, complete wast...
Predicted Rating: 1 stars

Review: Amazing quality, exceeded all my expectations, wil...
Predicted Rating: 5 stars

Review: It is okay, nothing special but does the job...
Predicted Rating: 3 stars


# Save RNN model

In [ ]:
# Save both models to Drive
model.save('/content/drive/MyDrive/cnn_sentiment_model.h5')
rnn_model.save('/content/drive/MyDrive/rnn_rating_model.h5')

print("Both models saved to Google Drive!")
print("CNN Sentiment Model → cnn_sentiment_model.h5")
print("RNN Rating Model    → rnn_rating_model.h5")

Both models saved to Google Drive!
CNN Sentiment Model → cnn_sentiment_model.h5
RNN Rating Model    → rnn_rating_model.h5


# Word2Vec Embeddings + SVM Classifier

In [ ]:
from gensim.models import Word2Vec
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

print("Libraries ready!")

Libraries ready!


# Train Word2Vec embeddings

In [ ]:
# Tokenize cleaned reviews into list of words
sentences = df['Cleaned_Text'].astype(str).apply(
    lambda x: x.split()
).tolist()

print("Training Word2Vec...")

word2vec_model = Word2Vec(
    sentences=sentences,
    vector_size=100,  # each word = 100 numbers
    window=5,         # looks 5 words left and right for context
    min_count=2,      # ignore words appearing less than 2 times
    workers=4,        # parallel processing
    epochs=5
)

print("Word2Vec Done!")
print("Vocabulary size:", len(word2vec_model.wv))

# Test — find similar words
print("\nWords similar to 'good':")
print(word2vec_model.wv.most_similar('good', topn=5))

print("\nWords similar to 'terrible':")
print(word2vec_model.wv.most_similar('terrible', topn=5))

Training Word2Vec...
Word2Vec Done!
Vocabulary size: 102909

Words similar to 'good':
[('great', 0.8068118691444397), ('decent', 0.7901737689971924), ('awesome', 0.6802608966827393), ('bad', 0.6725966930389404), ('fantastic', 0.6679133772850037)]

Words similar to 'terrible':
[('horrible', 0.9508200883865356), ('awful', 0.9165431261062622), ('disgusting', 0.822450578212738), ('horrid', 0.8139666318893433), ('gross', 0.7682281732559204)]


# Convert reviews to vectors

In [ ]:
# Each review = average of all its word vectors
# "good pizza taste" → average of 3 word vectors = 1 vector

def review_to_vector(review):
    words = review.split()
    # Only keep words that exist in Word2Vec vocabulary
    vectors = [
        word2vec_model.wv[word]
        for word in words
        if word in word2vec_model.wv
    ]
    if vectors:
        # Average all word vectors into one review vector
        return np.mean(vectors, axis=0)
    else:
        # Return zeros if no words found
        return np.zeros(100)

print("Converting reviews to vectors...")

X_w2v = np.array([
    review_to_vector(review)
    for review in df['Cleaned_Text'].astype(str)
])

print("Done! Shape:", X_w2v.shape)

Converting reviews to vectors...
Done! Shape: (393579, 100)


# Train SVM classifier

In [ ]:
# Use sentiment as target label
y_svm = df['Sentiment']

# Split data
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_w2v, y_svm,
    test_size=0.2,
    random_state=42
)

print("Training SVM classifier...")

# LinearSVC is fast SVM — perfect for large text datasets
svm_model = LinearSVC(max_iter=1000)
svm_model.fit(X_train_s, y_train_s)

print("SVM Training Done!")

# Test accuracy
y_pred = svm_model.predict(X_test_s)
print(f"\nSVM Accuracy: {accuracy_score(y_test_s, y_pred)*100:.2f}%")
print("\nDetailed Report:")
print(classification_report(y_test_s, y_pred))

Training SVM classifier...
SVM Training Done!

SVM Accuracy: 84.12%

Detailed Report:
              precision    recall  f1-score   support

    Negative       0.69      0.56      0.61     11360
     Neutral       0.46      0.01      0.01      5890
    Positive       0.86      0.97      0.91     61466

    accuracy                           0.84     78716
   macro avg       0.67      0.51      0.51     78716
weighted avg       0.81      0.84      0.80     78716



#Test with real reviews

In [ ]:
test_reviews = [
    "Absolutely love this product best purchase ever",
    "Complete garbage broke after one day total waste",
    "It is average nothing special just okay product"
]

for review in test_reviews:
    # Clean the review
    cleaned = clean_text(review)
    # Convert to vector
    vector = review_to_vector(cleaned).reshape(1, -1)
    # Predict
    prediction = svm_model.predict(vector)
    print(f"Review : {review[:45]}...")
    print(f"Predicted : {prediction[0]}\n")

Review : Absolutely love this product best purchase ev...
Predicted : Positive

Review : Complete garbage broke after one day total wa...
Predicted : Negative

Review : It is average nothing special just okay produ...
Predicted : Negative



#  Save everything

In [ ]:
import pickle

# Save Word2Vec model
word2vec_model.save('/content/drive/MyDrive/word2vec_model.model')

# Save SVM model
with open('/content/drive/MyDrive/svm_model.pkl', 'wb') as f:
    pickle.dump(svm_model, f)

# Save final dataset
df.to_csv('/content/drive/MyDrive/Reviews_Final.csv', index=False)

print("All saved to Google Drive!")
print("word2vec_model.model  → Word2Vec embeddings")
print("svm_model.pkl         → SVM classifier")
print("Reviews_Final.csv     → Complete dataset")

All saved to Google Drive!
word2vec_model.model  → Word2Vec embeddings
svm_model.pkl         → SVM classifier
Reviews_Final.csv     → Complete dataset


# Streamlit App


In [ ]:
!pip install streamlit pyngrok -q

print("Streamlit installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 48.8 MB/s eta 0:00:00
Streamlit installed!


# Create the app file

In [ ]:
app_code = '''
import streamlit as st
import numpy as np
import pickle
import re
import string
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from gensim.models import Word2Vec
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import pickle

nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

# ---- Page config ----
st.set_page_config(
    page_title="NLP Review Intelligence",
    page_icon="🧠",
    layout="centered"
)

# ---- Title ----
st.title("🧠 NLP Review Intelligence System")
st.markdown("### Amazon Review Analyser — CNN · RNN · SVM")
st.markdown("---")

# ---- Load models ----
@st.cache_resource
def load_all_models():
    cnn_model = load_model(
        "/content/drive/MyDrive/cnn_sentiment_model.h5"
    )
    rnn_model = load_model(
        "/content/drive/MyDrive/rnn_rating_model.h5"
    )
    with open("/content/drive/MyDrive/svm_model.pkl", "rb") as f:
        svm_model = pickle.load(f)
    w2v_model = Word2Vec.load(
        "/content/drive/MyDrive/word2vec_model.model"
    )
    with open(
        "/content/drive/MyDrive/tokenizer.pkl", "rb"
    ) as f:
        tokenizer = pickle.load(f)
    return cnn_model, rnn_model, svm_model, w2v_model, tokenizer

cnn_model, rnn_model, svm_model, w2v_model, tokenizer = (
    load_all_models()
)

# ---- Clean text function ----
def clean_text(text):
    stop_words = set(stopwords.words("english"))
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)
    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )
    text = re.sub(r"\\d+", "", text)
    tokens = word_tokenize(text)
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [w for w in tokens if len(w) > 2]
    return " ".join(tokens)

# ---- Review to vector ----
def review_to_vector(review):
    words = review.split()
    vectors = [
        w2v_model.wv[w]
        for w in words
        if w in w2v_model.wv
    ]
    if vectors:
        return np.mean(vectors, axis=0)
    return np.zeros(100)

# ---- Input box ----
st.markdown("#### 📝 Enter a product review below")
user_review = st.text_area(
    "",
    placeholder=(
        "Example: This product is amazing, "
        "best purchase I ever made!"
    ),
    height=150
)

# ---- Predict button ----
if st.button("🔍 Analyse Review", use_container_width=True):
    if user_review.strip() == "":
        st.warning("Please enter a review first!")
    else:
        cleaned = clean_text(user_review)
        MAX_LEN = 100

        # CNN Sentiment
        seq = tokenizer.texts_to_sequences([cleaned])
        pad = pad_sequences(seq, maxlen=MAX_LEN)
        cnn_pred = cnn_model.predict(pad, verbose=0)
        labels = ["Negative", "Neutral", "Positive"]
        sentiment = labels[np.argmax(cnn_pred)]
        confidence = float(np.max(cnn_pred)) * 100

        # RNN Rating
        rnn_pred = rnn_model.predict(pad, verbose=0)
        rating = np.argmax(rnn_pred) + 1

        # SVM
        vector = review_to_vector(cleaned).reshape(1, -1)
        svm_result = svm_model.predict(vector)[0]

        # ---- Display results ----
        st.markdown("---")
        st.markdown("### 📊 Analysis Results")

        col1, col2, col3 = st.columns(3)

        with col1:
            emoji = (
                "😊" if sentiment == "Positive"
                else "😡" if sentiment == "Negative"
                else "😐"
            )
            color = (
                "green" if sentiment == "Positive"
                else "red" if sentiment == "Negative"
                else "orange"
            )
            st.markdown(
                f"#### CNN Sentiment"
            )
            st.markdown(
                f"<h2 style='color:{color}'>"
                f"{emoji} {sentiment}</h2>",
                unsafe_allow_html=True
            )
            st.caption(f"Confidence: {confidence:.1f}%")

        with col2:
            stars = "⭐" * rating
            st.markdown("#### RNN Rating")
            st.markdown(
                f"<h2 style='color:#1A5276'>"
                f"{stars}</h2>",
                unsafe_allow_html=True
            )
            st.caption(f"Predicted: {rating} out of 5")

        with col3:
            st.markdown("#### SVM Result")
            st.markdown(
                f"<h2 style='color:#6C3483'>"
                f"{svm_result}</h2>",
                unsafe_allow_html=True
            )
            st.caption("Support Vector Machine")

        st.markdown("---")
        st.markdown("**Cleaned text used for analysis:**")
        st.code(cleaned)

st.markdown("---")
st.caption(
    "Built with CNN · RNN · SVM · Word2Vec · LDA "
    "| 393K Amazon Reviews"
)
'''

# Save the app
with open("app.py", "w") as f:
    f.write(app_code)

print("app.py created!")

app.py created!


# Save tokenizer first

In [ ]:
import pickle

# Save tokenizer — needed by Streamlit app
with open('/content/drive/MyDrive/tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

print("Tokenizer saved!")

Tokenizer saved!


# Run the app

In [ ]:
from pyngrok import ngrok

# Start Streamlit
!streamlit run app.py &

# Create public URL
public_url = ngrok.connect(8501)
print("Your app is live at:", public_url)




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.125.131.219:8501

  Stopping...


ERROR:pyngrok.process.ngrok:t=2026-04-26T09:08:24+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-04-26T09:08:24+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-04-26T09:08:24+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your aut

PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

In [ ]:
# First stop the old streamlit
import os
os.system("pkill -f streamlit")

# Set your ngrok token
!ngrok authtoken 3CtA1XlcyXbOvVw2ICAQDFk13ZW_7AvSnTiYJreeuWHFJH33e  # ← paste your token

print("Token set!")

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
Token set!


In [ ]:
from pyngrok import ngrok
import subprocess
import time

# Kill any existing tunnels
ngrok.kill()

# Start streamlit in background
subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.headless", "true"]
)

# Wait for it to start
time.sleep(5)

# Create public URL
tunnel = ngrok.connect(8501)
print("=" * 50)
print("✅ Your app is LIVE at:")
print(tunnel.public_url)
print("=" * 50)

✅ Your app is LIVE at:
https://mortality-handshake-extending.ngrok-free.dev


# completed
